# Epic 1 — Tutorial: Synthetic Voids on Your Own Images

> **Scope:** UDM **Epic 1** (X-ray void synthesis foundation). For **Epic 2** (GAN-based defects), see `nbs/epic2_00_gan_overview.ipynb`.

By default, this repo generates **fully synthetic** X-ray images using Beer-Lambert physics.
But in practice, you often have **real normal images** (no voids) and want to overlay synthetic voids
to get paired `(image, mask)` data for training a segmentation model.

This tutorial shows how to do exactly that.

## How it works

The `SyntheticSampleGenerator.generate()` method accepts an optional `background` parameter:

- **`background=None`** (default) — generates a synthetic background via Beer-Lambert physics
- **`background=your_image`** — uses your real image as the background, then overlays synthetic voids

When you pass a real image, the generator:
1. Resizes it to the target resolution (e.g. 512x512)
2. Normalizes it to float32 [0, 1] via p2/p98 percentile normalization
3. Generates random void shapes and inserts them using Beer-Lambert contrast
4. Returns the composited image + binary mask

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic1 import SyntheticSampleGenerator, GeneratorConfig

## 1. Single image — quick demo

Load one of your normal images and pass it as `background`.

In [ ]:
# --- Replace this block with your own image loading ---
# For this demo we create a fake "real" background that looks like
# a solder joint X-ray. Replace with:
#   real_bg = cv2.imread("path/to/your/normal_image.png", cv2.IMREAD_UNCHANGED)

rng = np.random.default_rng(0)
H, W = 512, 512
# Simulate a dark solder region with slight spatial variation
real_bg = (0.3 + 0.05 * rng.standard_normal((H, W))).astype(np.float32)
real_bg = np.clip(real_bg, 0.0, 1.0)

print(f"Background shape: {real_bg.shape}, dtype: {real_bg.dtype}")

In [ ]:
cfg = GeneratorConfig(height=H, width=W)
gen = SyntheticSampleGenerator(cfg, seed=42)

# Pass your real image as background
image, mask, meta = gen.generate(
    image_id="real_bg_sample_001",
    background=real_bg,
)

print(f"Output image: {image.shape}, dtype: {image.dtype}")
print(f"Output mask:  {mask.shape},  dtype: {mask.dtype}")
print(f"Voids placed: {meta.n_voids}")
print(f"Total void area: {meta.total_void_area_fraction:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(real_bg, cmap='gray')
axes[0].set_title('Original (no voids)')

# image is uint16, normalize for display
axes[1].imshow(image.astype(np.float32) / 65535.0, cmap='gray')
axes[1].set_title(f'With synthetic voids ({meta.n_voids} voids)')

axes[2].imshow(mask, cmap='gray')
axes[2].set_title('Ground truth mask')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Batch generation from a folder of real images

If you have a directory of normal images, you can loop through them
and generate multiple synthetic samples per image.

In [ ]:
def generate_from_real_backgrounds(
    backgrounds_dir: str,
    output_dir: str,
    samples_per_image: int = 5,
    height: int = 512,
    width: int = 512,
    seed: int = 42,
):
    """
    Generate synthetic void images using real backgrounds.

    Args:
        backgrounds_dir: folder with normal (void-free) images
        output_dir:      where to save image/mask pairs
        samples_per_image: number of synthetic variants per background
        height, width:   output resolution
        seed:            for reproducibility
    """
    bg_dir = Path(backgrounds_dir)
    out = Path(output_dir)
    (out / 'images').mkdir(parents=True, exist_ok=True)
    (out / 'masks').mkdir(parents=True, exist_ok=True)

    # Collect background image paths
    extensions = {'.png', '.tif', '.tiff', '.jpg', '.jpeg', '.bmp'}
    bg_paths = sorted(
        p for p in bg_dir.iterdir()
        if p.suffix.lower() in extensions
    )
    print(f"Found {len(bg_paths)} background images in {bg_dir}")

    cfg = GeneratorConfig(height=height, width=width)
    results = []
    sample_idx = 0

    for bg_path in bg_paths:
        # Load real image (grayscale, preserve bit depth)
        bg = cv2.imread(str(bg_path), cv2.IMREAD_UNCHANGED)
        if bg is None:
            print(f"  Skipping {bg_path.name} (could not read)")
            continue
        # Convert RGB to grayscale if needed
        if bg.ndim == 3:
            bg = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)

        for variant in range(samples_per_image):
            # Each variant gets a unique seed
            gen = SyntheticSampleGenerator(cfg, seed=seed + sample_idx * 1000)
            image_id = f"sample_{sample_idx:06d}"

            image, mask, meta = gen.generate(
                image_id=image_id,
                background=bg,
            )

            # Save
            cv2.imwrite(str(out / 'images' / f'{image_id}.png'), image)
            cv2.imwrite(str(out / 'masks'  / f'{image_id}.png'), mask)

            results.append(meta.to_dict())
            sample_idx += 1

    print(f"Generated {sample_idx} samples -> {out}")
    return results

In [ ]:
# --- Uncomment and adjust paths to run on your data ---
# results = generate_from_real_backgrounds(
#     backgrounds_dir="data/raw_backgrounds",
#     output_dir="data/synthetic_from_real",
#     samples_per_image=5,
#     height=512,
#     width=512,
#     seed=42,
# )

## 3. Tuning void appearance

You can control how the synthetic voids look via `GeneratorConfig`.

In [ ]:
from udm_epic1.physics.beer_lambert import BeerLambertConfig

# Example: fewer but larger voids, higher contrast
cfg_custom = GeneratorConfig(
    height=512,
    width=512,
    void_count_range=(1, 3),         # fewer voids per image
    min_area_fraction=0.01,          # minimum 1% of image
    max_area_fraction=0.15,          # up to 15%
    empty_image_fraction=0.0,        # no empty images
    shape_weights={
        'ellipse': 0.6,              # mostly round voids
        'irregular_blob': 0.3,
        'elongated': 0.05,
        'cluster': 0.05,
    },
    physics=BeerLambertConfig(
        apply_sft_correction=False,  # skip if your images are already corrected
    ),
)

gen_custom = SyntheticSampleGenerator(cfg_custom, seed=99)
image2, mask2, meta2 = gen_custom.generate(
    image_id='custom_voids',
    background=real_bg,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image2.astype(np.float32) / 65535.0, cmap='gray')
axes[0].set_title(f'Custom config ({meta2.n_voids} voids)')
axes[1].imshow(mask2, cmap='gray')
axes[1].set_title('Mask')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Visual grid — multiple variants from one background

Different seeds produce different void placements, shapes, and counts
from the same background image.

In [ ]:
n_variants = 6
cfg = GeneratorConfig(height=H, width=W)

fig, axes = plt.subplots(2, n_variants, figsize=(3 * n_variants, 6))

for i in range(n_variants):
    gen_i = SyntheticSampleGenerator(cfg, seed=i * 1000)
    img_i, msk_i, meta_i = gen_i.generate(
        image_id=f'variant_{i}',
        background=real_bg,
    )
    axes[0, i].imshow(img_i.astype(np.float32) / 65535.0, cmap='gray')
    axes[0, i].set_title(f'seed={i * 1000}\n{meta_i.n_voids} voids')
    axes[0, i].axis('off')

    axes[1, i].imshow(msk_i, cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Mask', fontsize=12)
plt.suptitle('Multiple synthetic variants from the same background', fontsize=14)
plt.tight_layout()
plt.show()

## Tips

- **Input format**: Your images can be 8-bit, 16-bit, or float32. The generator handles conversion automatically.
- **Resolution**: Images are resized to `(height, width)` from the config. Use a resolution close to your originals to avoid artifacts.
- **SFT correction**: If your images already have background correction applied, set `apply_sft_correction=False` in `BeerLambertConfig` to avoid double-correction.
- **Samples per image**: Generate 5-10 variants per background to maximize data diversity without needing thousands of real images.
- **Augmentation**: The augmentation pipeline (`udm_epic1.augmentation`) is designed for training-time use. Don't save augmented images to disk.